# 03 Single-Market Mini-Pipeline

This notebook builds an initial end-to-end mini-pipeline for the weather forecasting and Polymarket trading project.

The aim is to connect:

1. Polymarket temperature-bin market data;
2. weather forecast data;
3. a simple predictive temperature distribution;
4. model-implied bin probabilities;
5. market-implied bin probabilities;
6. discrepancy and scoring analysis.

This notebook is intended as a methodological prototype for one market, rather than the final forecasting model or final trading strategy.

In [2]:
import requests
import pandas as pd
import json
import os
import re
import math
from datetime import datetime, timezone, timedelta
from pprint import pprint

GAMMA_BASE = "https://gamma-api.polymarket.com"
CLOB_BASE = "https://clob.polymarket.com"
ARCHIVE_BASE = "https://archive-api.open-meteo.com/v1/archive"
PREVIOUS_RUNS_BASE = "https://previous-runs-api.open-meteo.com/v1/forecast"

def get_json(url, params=None, timeout=30):
    response = requests.get(url, params=params, timeout=timeout)
    print("URL:", response.url)
    print("Status code:", response.status_code)
    response.raise_for_status()
    return response.json()

def pretty(obj, max_chars=3000):
    text = json.dumps(obj, indent=2, ensure_ascii=False)
    print(text[:max_chars])
    if len(text) > max_chars:
        print(f"\n... truncated, total length = {len(text)} characters")

def parse_jsonish(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return []
    return []

## 1. Select one sample temperature event

The sample event is a Hong Kong daily maximum temperature market.

Using one known event keeps the notebook focused on the data-processing logic: extracting Polymarket probabilities, retrieving forecast data, mapping a forecast distribution into market bins, and comparing model-implied probabilities with market-implied probabilities.

In [4]:
event_slug = "highest-temperature-in-hong-kong-on-june-10-2026"
event_date = "2026-06-10"

event_data = get_json(f"{GAMMA_BASE}/events", params={"slug": event_slug})

if isinstance(event_data, list):
    event = event_data[0]
else:
    event = event_data

print("Event title:", event.get("title"))
print("Event ID:", event.get("id"))
print("Event endDate:", event.get("endDate"))
print("Number of markets:", len(event.get("markets", [])))

URL: https://gamma-api.polymarket.com/events?slug=highest-temperature-in-hong-kong-on-june-10-2026
Status code: 200
Event title: Highest temperature in Hong Kong on June 10?
Event ID: 571434
Event endDate: 2026-06-10T12:00:00Z
Number of markets: 11


## 2. Extract Polymarket YES prices and temperature bins

A temperature-bin event on Polymarket is represented as multiple binary YES/NO markets.

For this project, the relevant market-implied probability for each bin is the YES price. The YES prices across bins can then be interpreted as an implied probability distribution over temperature ranges, subject to market microstructure effects, bid-ask spreads, rounding and market closure.

In [6]:
markets = event.get("markets", [])
rows = []

for m in markets:
    outcomes = parse_jsonish(m.get("outcomes"))
    prices = parse_jsonish(m.get("outcomePrices"))
    token_ids = parse_jsonish(m.get("clobTokenIds"))
    
    for i, outcome in enumerate(outcomes):
        rows.append({
            "event_slug": event_slug,
            "event_title": event.get("title"),
            "market_id": m.get("id"),
            "market_question": m.get("question"),
            "market_slug": m.get("slug"),
            "condition_id": m.get("conditionId"),
            "outcome_index": i,
            "outcome": outcome,
            "outcome_price": float(prices[i]) if i < len(prices) and prices[i] is not None else None,
            "clob_token_id": token_ids[i] if i < len(token_ids) else None,
            "active": m.get("active"),
            "closed": m.get("closed"),
            "volume": m.get("volume"),
        })

outcome_df = pd.DataFrame(rows)

yes_df = outcome_df[
    outcome_df["outcome"].astype(str).str.lower() == "yes"
].copy()

yes_df = yes_df.sort_values("market_id").reset_index(drop=True)

yes_df[[
    "market_id",
    "market_question",
    "outcome_price",
    "clob_token_id",
    "closed",
    "volume",
]]

,market_id,market_question,outcome_price,clob_token_id,closed,volume
0,2467879,Will the highest temperature in Hong Kong be 2...,0.0,7511480192414931839332517660053113305812541935...,True,2459.0020000000004
1,2467880,Will the highest temperature in Hong Kong be 2...,0.0,7138240848352469602092737651578682324028832907...,True,4339.706145
2,2467881,Will the highest temperature in Hong Kong be 2...,0.0,6242282160040905745969830144723528288889057744...,True,14542.874648999994
3,2467882,Will the highest temperature in Hong Kong be 2...,0.0,1571864177641136098347889516528979652783697133...,True,39412.588582000026
4,2467883,Will the highest temperature in Hong Kong be 2...,1.0,1039526083584639149999855318189075032653832204...,True,79509.85583500004
5,2467884,Will the highest temperature in Hong Kong be 2...,0.0,1639766626668914739093438255816708877446284743...,True,42174.46050599999
6,2467885,Will the highest temperature in Hong Kong be 3...,0.0,6069572569713038731920992570474313072224509346...,True,52907.91220299997
7,2467886,Will the highest temperature in Hong Kong be 3...,0.0,4031083886066835253354353137248149381722677087...,True,46021.93409500009
8,2467887,Will the highest temperature in Hong Kong be 3...,0.0,3206079022797271009184189942960716010997695920...,True,40645.79464400003
9,2467888,Will the highest temperature in Hong Kong be 3...,0.0,1804287360424248099758232744185023997430032254...,True,14856.569109000007


In [7]:
def extract_temperature_phrase(question):
    q = re.sub(r"\s+", " ", str(question))
    
    if " be " in q and " on " in q:
        return q.split(" be ", 1)[1].split(" on ", 1)[0].strip()
    
    return q

def parse_temperature_bin(question):
    phrase = extract_temperature_phrase(question)
    phrase_lower = phrase.lower()
    
    numbers = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", phrase)]
    
    lower = None
    upper = None
    
    if len(numbers) == 0:
        lower, upper = None, None
    
    elif any(key in phrase_lower for key in ["or lower", "or less", "or below", "below", "under", "less than"]):
        # For integer-labelled bins, "24°C or below" is treated as all temperatures below 25°C.
        lower, upper = -math.inf, numbers[0] + 1.0
    
    elif any(key in phrase_lower for key in ["or higher", "or more", "or above", "above", "over", "greater than"]):
        # "34°C or higher" is treated as temperatures from 34°C upwards.
        lower, upper = numbers[0], math.inf
    
    elif len(numbers) >= 2:
        # If the question explicitly gives two bounds, use them directly.
        lower, upper = numbers[0], numbers[1]
    
    elif len(numbers) == 1:
        # A single integer label such as "28°C" is treated as the one-degree bin [28, 29).
        lower, upper = numbers[0], numbers[0] + 1.0
    
    if lower == -math.inf:
        bin_label = f"< {upper:.1f}"
    elif upper == math.inf:
        bin_label = f">= {lower:.1f}"
    elif lower is None or upper is None:
        bin_label = "unparsed"
    else:
        bin_label = f"[{lower:.1f}, {upper:.1f})"
    
    return pd.Series({
        "temperature_phrase": phrase,
        "bin_lower": lower,
        "bin_upper": upper,
        "bin_label": bin_label,
    })

bin_df = yes_df["market_question"].apply(parse_temperature_bin)
yes_df = pd.concat([yes_df, bin_df], axis=1)

yes_df = yes_df.sort_values(["bin_lower", "bin_upper"]).reset_index(drop=True)

yes_df[[
    "market_question",
    "temperature_phrase",
    "bin_lower",
    "bin_upper",
    "bin_label",
    "outcome_price",
]]

,market_question,temperature_phrase,bin_lower,bin_upper,bin_label,outcome_price
0,Will the highest temperature in Hong Kong be 2...,24°C or below,-inf,25.0,< 25.0,0.0
1,Will the highest temperature in Hong Kong be 2...,25°C,25.0,26.0,"[25.0, 26.0)",0.0
2,Will the highest temperature in Hong Kong be 2...,26°C,26.0,27.0,"[26.0, 27.0)",0.0
3,Will the highest temperature in Hong Kong be 2...,27°C,27.0,28.0,"[27.0, 28.0)",0.0
4,Will the highest temperature in Hong Kong be 2...,28°C,28.0,29.0,"[28.0, 29.0)",1.0
5,Will the highest temperature in Hong Kong be 2...,29°C,29.0,30.0,"[29.0, 30.0)",0.0
6,Will the highest temperature in Hong Kong be 3...,30°C,30.0,31.0,"[30.0, 31.0)",0.0
7,Will the highest temperature in Hong Kong be 3...,31°C,31.0,32.0,"[31.0, 32.0)",0.0
8,Will the highest temperature in Hong Kong be 3...,32°C,32.0,33.0,"[32.0, 33.0)",0.0
9,Will the highest temperature in Hong Kong be 3...,33°C,33.0,34.0,"[33.0, 34.0)",0.0


In [8]:
print("Number of YES bins:", len(yes_df))
print("Sum of current YES prices:", yes_df["outcome_price"].sum())

if yes_df["bin_lower"].isna().any() or yes_df["bin_upper"].isna().any():
    print("Warning: at least one bin was not parsed correctly. Inspect market_question manually.")
else:
    print("All bins parsed into numerical lower/upper bounds.")

Number of YES bins: 11
Sum of current YES prices: 1.0
All bins parsed into numerical lower/upper bounds.


### Bin convention used in this notebook

The Hong Kong market uses integer-labelled temperature bins. In this notebook, a label such as `28°C` is interpreted as the interval `[28.0, 29.0)`, while `24°C or below` is treated as temperatures below `25.0°C`, and `34°C or higher` is treated as temperatures greater than or equal to `34.0°C`.

This convention is used to make the continuous predictive temperature distribution comparable with the discrete Polymarket outcome bins. The exact settlement-source rule should still be checked against the contract text and Hong Kong Observatory data in the full analysis.

## 3. Retrieve market-implied probabilities at a historical comparison time

The current YES prices of a resolved market may reflect the final outcome rather than pre-settlement beliefs.

For a meaningful model-versus-market comparison, this notebook retrieves historical price paths for each YES token and selects the latest observed price at or before a chosen comparison time.

The comparison time is set to approximately one day before the event end time.

In [11]:
event_end_utc = pd.to_datetime(event.get("endDate"), utc=True)
comparison_time_utc = event_end_utc - pd.Timedelta(days=1)

print("Event end time:", event_end_utc)
print("Comparison time:", comparison_time_utc)

Event end time: 2026-06-10 12:00:00+00:00
Comparison time: 2026-06-09 12:00:00+00:00


In [12]:
def fetch_price_history(token_id):
    data = get_json(
        f"{CLOB_BASE}/prices-history",
        params={
            "market": str(token_id),
            "interval": "max",
        }
    )
    
    hist = data.get("history", [])
    if len(hist) == 0:
        return pd.DataFrame(columns=["datetime_utc", "p", "t", "clob_token_id"])
    
    df = pd.DataFrame(hist)
    df["datetime_utc"] = pd.to_datetime(df["t"], unit="s", utc=True)
    df["p"] = pd.to_numeric(df["p"], errors="coerce")
    df["clob_token_id"] = str(token_id)
    return df[["datetime_utc", "p", "t", "clob_token_id"]]

all_histories = []

for _, row in yes_df.iterrows():
    token_id = row["clob_token_id"]
    
    try:
        hist_df = fetch_price_history(token_id)
        hist_df["market_id"] = row["market_id"]
        hist_df["bin_label"] = row["bin_label"]
        all_histories.append(hist_df)
    except Exception as e:
        print("History failed for token:", token_id)
        print(e)

price_history_df = pd.concat(all_histories, ignore_index=True) if len(all_histories) > 0 else pd.DataFrame()

print("Total historical price observations:", len(price_history_df))
price_history_df.head()

URL: https://clob.polymarket.com/prices-history?market=75114801924149318393325176600531133058125419354659304337062518011403869315671&interval=max
Status code: 200
URL: https://clob.polymarket.com/prices-history?market=71382408483524696020927376515786823240288329073616677127679194797612102154666&interval=max
Status code: 200
URL: https://clob.polymarket.com/prices-history?market=62422821600409057459698301447235282888890577446869542091777011845356809868287&interval=max
Status code: 200
URL: https://clob.polymarket.com/prices-history?market=15718641776411360983478895165289796527836971336383880138777981093993104520784&interval=max
Status code: 200
URL: https://clob.polymarket.com/prices-history?market=103952608358463914999985531818907503265383220402407534332311099994541518367084&interval=max
Status code: 200
URL: https://clob.polymarket.com/prices-history?market=16397666266689147390934382558167088774462847438957547963956284647658172511927&interval=max
Status code: 200
URL: https://clob.pol

,datetime_utc,p,t,clob_token_id,market_id,bin_label
0,2026-06-08 04:10:06+00:00,0.010,1780891806,7511480192414931839332517660053113305812541935...,2467879,< 25.0
1,2026-06-08 04:20:04+00:00,0.005,1780892404,7511480192414931839332517660053113305812541935...,2467879,< 25.0
2,2026-06-08 04:30:08+00:00,0.005,1780893008,7511480192414931839332517660053113305812541935...,2467879,< 25.0
3,2026-06-08 04:40:04+00:00,0.005,1780893604,7511480192414931839332517660053113305812541935...,2467879,< 25.0
4,2026-06-08 04:50:07+00:00,0.005,1780894207,7511480192414931839332517660053113305812541935...,2467879,< 25.0


In [13]:
market_probs_at_time = []

for _, row in yes_df.iterrows():
    token_id = str(row["clob_token_id"])
    token_hist = price_history_df[price_history_df["clob_token_id"] == token_id].copy()
    
    token_hist_before = token_hist[token_hist["datetime_utc"] <= comparison_time_utc]
    
    if len(token_hist_before) > 0:
        selected = token_hist_before.sort_values("datetime_utc").iloc[-1]
        market_price = selected["p"]
        price_time = selected["datetime_utc"]
        price_source = "historical_price_at_or_before_comparison_time"
    else:
        market_price = row["outcome_price"]
        price_time = pd.NaT
        price_source = "fallback_current_outcome_price"
    
    market_probs_at_time.append({
        "market_id": row["market_id"],
        "bin_label": row["bin_label"],
        "bin_lower": row["bin_lower"],
        "bin_upper": row["bin_upper"],
        "market_question": row["market_question"],
        "market_probability_raw": market_price,
        "market_price_timestamp_utc": price_time,
        "market_price_source": price_source,
    })

market_prob_df = pd.DataFrame(market_probs_at_time)

raw_sum = market_prob_df["market_probability_raw"].sum()
market_prob_df["market_probability_normalised"] = (
    market_prob_df["market_probability_raw"] / raw_sum
    if raw_sum and raw_sum > 0 else None
)

print("Raw market probability sum:", raw_sum)

market_prob_df[[
    "bin_label",
    "market_probability_raw",
    "market_probability_normalised",
    "market_price_timestamp_utc",
    "market_price_source",
]]

Raw market probability sum: 1.0665


,bin_label,market_probability_raw,market_probability_normalised,market_price_timestamp_utc,market_price_source
0,< 25.0,0.0015,0.001406,2026-06-09 11:50:04+00:00,historical_price_at_or_before_comparison_time
1,"[25.0, 26.0)",0.0035,0.003282,2026-06-09 11:50:04+00:00,historical_price_at_or_before_comparison_time
2,"[26.0, 27.0)",0.0255,0.023910,2026-06-09 11:50:04+00:00,historical_price_at_or_before_comparison_time
3,"[27.0, 28.0)",0.0980,0.091889,2026-06-09 11:50:04+00:00,historical_price_at_or_before_comparison_time
4,"[28.0, 29.0)",0.2700,0.253165,2026-06-09 11:50:04+00:00,historical_price_at_or_before_comparison_time
5,"[29.0, 30.0)",0.3250,0.304735,2026-06-09 11:50:04+00:00,historical_price_at_or_before_comparison_time
6,"[30.0, 31.0)",0.2350,0.220347,2026-06-09 11:50:04+00:00,historical_price_at_or_before_comparison_time
7,"[31.0, 32.0)",0.0750,0.070323,2026-06-09 11:50:04+00:00,historical_price_at_or_before_comparison_time
8,"[32.0, 33.0)",0.0245,0.022972,2026-06-09 11:50:04+00:00,historical_price_at_or_before_comparison_time
9,"[33.0, 34.0)",0.0050,0.004688,2026-06-09 11:50:04+00:00,historical_price_at_or_before_comparison_time


### Market-probability normalisation

At the selected comparison time, the raw YES prices across bins sum to slightly above one. This is expected because the event is represented by multiple binary markets and the observed prices may reflect bid-ask spreads, liquidity effects, rounding and market microstructure noise.

For the probability comparison below, the raw YES prices are normalised across bins so that they can be compared with the model-implied probability distribution.

## 4. Retrieve weather forecast and realised/proxy-realised temperature

Open-Meteo is used here as a convenient weather-data proxy for testing the pipeline.

This is not yet the final settlement source for the Hong Kong contract. The contract refers to Hong Kong Observatory data, so exact settlement-source matching remains a separate data task.

In [16]:
hk_latitude = 22.3022
hk_longitude = 114.1746
hk_timezone = "Asia/Hong_Kong"

hk_archive = get_json(
    ARCHIVE_BASE,
    params={
        "latitude": hk_latitude,
        "longitude": hk_longitude,
        "start_date": event_date,
        "end_date": event_date,
        "daily": "temperature_2m_max",
        "hourly": "temperature_2m",
        "timezone": hk_timezone,
        "temperature_unit": "celsius",
    }
)

realised_proxy_temp = hk_archive["daily"]["temperature_2m_max"][0]
print("Open-Meteo proxy-realised daily max:", realised_proxy_temp)

URL: https://archive-api.open-meteo.com/v1/archive?latitude=22.3022&longitude=114.1746&start_date=2026-06-10&end_date=2026-06-10&daily=temperature_2m_max&hourly=temperature_2m&timezone=Asia%2FHong_Kong&temperature_unit=celsius
Status code: 200
Open-Meteo proxy-realised daily max: 27.4


In [17]:
previous_run_vars = ["temperature_2m"] + [
    f"temperature_2m_previous_day{i}" for i in range(1, 8)
]

hk_prev_runs = get_json(
    PREVIOUS_RUNS_BASE,
    params={
        "latitude": hk_latitude,
        "longitude": hk_longitude,
        "start_date": event_date,
        "end_date": event_date,
        "hourly": ",".join(previous_run_vars),
        "timezone": hk_timezone,
        "temperature_unit": "celsius",
    }
)

hk_prev_hourly_df = pd.DataFrame(hk_prev_runs["hourly"])
hk_prev_hourly_df["time"] = pd.to_datetime(hk_prev_hourly_df["time"])

def extract_lead_time_days(variable_name):
    if variable_name == "temperature_2m":
        return 0
    match = re.search(r"previous_day(\d+)", variable_name)
    return int(match.group(1)) if match else None

leadtime_rows = []

for col in previous_run_vars:
    if col in hk_prev_hourly_df.columns:
        daily_max = pd.to_numeric(hk_prev_hourly_df[col], errors="coerce").max()
        leadtime_rows.append({
            "city": "Hong Kong",
            "event_date": event_date,
            "forecast_variable": col,
            "lead_time_days_before_valid_time": extract_lead_time_days(col),
            "forecast_daily_max": daily_max,
            "realised_proxy_daily_max": realised_proxy_temp,
            "forecast_error": daily_max - realised_proxy_temp,
            "unit": "celsius",
            "source": "Open-Meteo Previous Runs API",
        })

weather_forecast_df = pd.DataFrame(leadtime_rows).sort_values("lead_time_days_before_valid_time")

weather_forecast_df

URL: https://previous-runs-api.open-meteo.com/v1/forecast?latitude=22.3022&longitude=114.1746&start_date=2026-06-10&end_date=2026-06-10&hourly=temperature_2m%2Ctemperature_2m_previous_day1%2Ctemperature_2m_previous_day2%2Ctemperature_2m_previous_day3%2Ctemperature_2m_previous_day4%2Ctemperature_2m_previous_day5%2Ctemperature_2m_previous_day6%2Ctemperature_2m_previous_day7&timezone=Asia%2FHong_Kong&temperature_unit=celsius
Status code: 200


,city,event_date,forecast_variable,lead_time_days_before_valid_time,forecast_daily_max,realised_proxy_daily_max,forecast_error,unit,source
0,Hong Kong,2026-06-10,temperature_2m,0,27.4,27.4,0.0,celsius,Open-Meteo Previous Runs API
1,Hong Kong,2026-06-10,temperature_2m_previous_day1,1,28.2,27.4,0.8,celsius,Open-Meteo Previous Runs API
2,Hong Kong,2026-06-10,temperature_2m_previous_day2,2,30.0,27.4,2.6,celsius,Open-Meteo Previous Runs API
3,Hong Kong,2026-06-10,temperature_2m_previous_day3,3,29.8,27.4,2.4,celsius,Open-Meteo Previous Runs API
4,Hong Kong,2026-06-10,temperature_2m_previous_day4,4,27.9,27.4,0.5,celsius,Open-Meteo Previous Runs API
5,Hong Kong,2026-06-10,temperature_2m_previous_day5,5,28.5,27.4,1.1,celsius,Open-Meteo Previous Runs API
6,Hong Kong,2026-06-10,temperature_2m_previous_day6,6,26.0,27.4,-1.4,celsius,Open-Meteo Previous Runs API
7,Hong Kong,2026-06-10,temperature_2m_previous_day7,7,26.6,27.4,-0.8,celsius,Open-Meteo Previous Runs API


## 5. Convert one forecast into a predictive temperature distribution

For this initial pipeline, the event temperature is modelled using a simple normal predictive distribution:

`T ~ Normal(mu, sigma^2)`

where:

- `mu` is the Open-Meteo forecast daily maximum for the chosen lead time;
- `sigma` is an assumed forecast uncertainty.

This is a baseline modelling assumption. In the full dissertation, the forecast uncertainty should be estimated from historical forecast errors and may depend on city, lead time, weather regime and data source.

In [19]:
selected_lead_time = 1
sigma = 1.5

selected_forecast = weather_forecast_df[
    weather_forecast_df["lead_time_days_before_valid_time"] == selected_lead_time
].iloc[0]

mu = selected_forecast["forecast_daily_max"]

print("Selected lead time:", selected_lead_time, "day before valid date")
print("Forecast mean mu:", mu)
print("Assumed sigma:", sigma)
print("Proxy-realised daily max:", realised_proxy_temp)

Selected lead time: 1 day before valid date
Forecast mean mu: 28.2
Assumed sigma: 1.5
Proxy-realised daily max: 27.4


In [20]:
def normal_cdf(x, mu, sigma):
    if x == -math.inf:
        return 0.0
    if x == math.inf:
        return 1.0
    z = (x - mu) / (sigma * math.sqrt(2))
    return 0.5 * (1 + math.erf(z))

def bin_probability_normal(lower, upper, mu, sigma):
    if lower is None or upper is None:
        return None
    return normal_cdf(upper, mu, sigma) - normal_cdf(lower, mu, sigma)

model_prob_df = market_prob_df.copy()

model_prob_df["model_probability_raw"] = model_prob_df.apply(
    lambda row: bin_probability_normal(
        row["bin_lower"],
        row["bin_upper"],
        mu,
        sigma,
    ),
    axis=1
)

model_sum = model_prob_df["model_probability_raw"].sum()

model_prob_df["model_probability_normalised"] = (
    model_prob_df["model_probability_raw"] / model_sum
    if model_sum and model_sum > 0 else None
)

print("Raw model probability sum:", model_sum)

model_prob_df[[
    "bin_label",
    "bin_lower",
    "bin_upper",
    "model_probability_raw",
    "model_probability_normalised",
]]

Raw model probability sum: 1.0


,bin_label,bin_lower,bin_upper,model_probability_raw,model_probability_normalised
0,< 25.0,-inf,25.0,0.016449,0.016449
1,"[25.0, 26.0)",25.0,26.0,0.054785,0.054785
2,"[26.0, 27.0)",26.0,27.0,0.140622,0.140622
3,"[27.0, 28.0)",27.0,28.0,0.235109,0.235109
4,"[28.0, 29.0)",28.0,29.0,0.256134,0.256134
5,"[29.0, 30.0)",29.0,30.0,0.181832,0.181832
6,"[30.0, 31.0)",30.0,31.0,0.084096,0.084096
7,"[31.0, 32.0)",31.0,32.0,0.025325,0.025325
8,"[32.0, 33.0)",32.0,33.0,0.004962,0.004962
9,"[33.0, 34.0)",33.0,34.0,0.000632,0.000632


## 6. Compare model-implied and market-implied probabilities

The table below is the main output of this notebook.

A positive discrepancy means that the weather model assigns more probability to a bin than the market does.

A negative discrepancy means that the market assigns more probability to a bin than the model does.

In [22]:
comparison_pipeline_df = model_prob_df.copy()

comparison_pipeline_df["model_minus_market"] = (
    comparison_pipeline_df["model_probability_normalised"]
    - comparison_pipeline_df["market_probability_normalised"]
)

comparison_pipeline_df["absolute_discrepancy"] = comparison_pipeline_df[
    "model_minus_market"
].abs()

comparison_pipeline_df = comparison_pipeline_df.sort_values("bin_lower").reset_index(drop=True)

comparison_pipeline_df[[
    "bin_label",
    "market_probability_raw",
    "market_probability_normalised",
    "model_probability_raw",
    "model_probability_normalised",
    "model_minus_market",
    "absolute_discrepancy",
    "market_price_timestamp_utc",
]]

,bin_label,market_probability_raw,market_probability_normalised,model_probability_raw,model_probability_normalised,model_minus_market,absolute_discrepancy,market_price_timestamp_utc
0,< 25.0,0.0015,0.001406,0.016449,0.016449,0.015042,0.015042,2026-06-09 11:50:04+00:00
1,"[25.0, 26.0)",0.0035,0.003282,0.054785,0.054785,0.051503,0.051503,2026-06-09 11:50:04+00:00
2,"[26.0, 27.0)",0.0255,0.023910,0.140622,0.140622,0.116712,0.116712,2026-06-09 11:50:04+00:00
3,"[27.0, 28.0)",0.0980,0.091889,0.235109,0.235109,0.143220,0.143220,2026-06-09 11:50:04+00:00
4,"[28.0, 29.0)",0.2700,0.253165,0.256134,0.256134,0.002969,0.002969,2026-06-09 11:50:04+00:00
5,"[29.0, 30.0)",0.3250,0.304735,0.181832,0.181832,-0.122903,0.122903,2026-06-09 11:50:04+00:00
6,"[30.0, 31.0)",0.2350,0.220347,0.084096,0.084096,-0.136251,0.136251,2026-06-09 11:50:04+00:00
7,"[31.0, 32.0)",0.0750,0.070323,0.025325,0.025325,-0.044999,0.044999,2026-06-09 11:50:04+00:00
8,"[32.0, 33.0)",0.0245,0.022972,0.004962,0.004962,-0.018010,0.018010,2026-06-09 11:50:04+00:00
9,"[33.0, 34.0)",0.0050,0.004688,0.000632,0.000632,-0.004056,0.004056,2026-06-09 11:50:04+00:00


## 7. Proxy realised-bin check and simple scoring

The realised-bin indicator in this notebook is based on the Open-Meteo realised/proxy-realised temperature.

This is useful for checking the mechanics of the scoring pipeline, but it should not be treated as the final settlement truth until Hong Kong Observatory data is retrieved and matched.

In [24]:
def contains_temperature(temp, lower, upper):
    if lower == -math.inf:
        return temp < upper
    if upper == math.inf:
        return temp >= lower
    return (temp >= lower) and (temp < upper)

comparison_pipeline_df["proxy_realised_indicator"] = comparison_pipeline_df.apply(
    lambda row: int(contains_temperature(realised_proxy_temp, row["bin_lower"], row["bin_upper"])),
    axis=1
)

model_brier_multiclass = (
    (comparison_pipeline_df["model_probability_normalised"] - comparison_pipeline_df["proxy_realised_indicator"]) ** 2
).sum()

market_brier_multiclass = (
    (comparison_pipeline_df["market_probability_normalised"] - comparison_pipeline_df["proxy_realised_indicator"]) ** 2
).sum()

eps = 1e-12

realised_row = comparison_pipeline_df[
    comparison_pipeline_df["proxy_realised_indicator"] == 1
]

if len(realised_row) > 0:
    model_log_score = -math.log(float(realised_row["model_probability_normalised"].iloc[0]) + eps)
    market_log_score = -math.log(float(realised_row["market_probability_normalised"].iloc[0]) + eps)
    realised_bin = realised_row["bin_label"].iloc[0]
else:
    model_log_score = None
    market_log_score = None
    realised_bin = None

score_summary = pd.DataFrame([{
    "event_slug": event_slug,
    "event_date": event_date,
    "comparison_time_utc": comparison_time_utc,
    "selected_forecast_lead_time_days": selected_lead_time,
    "forecast_mu": mu,
    "assumed_sigma": sigma,
    "realised_proxy_temperature": realised_proxy_temp,
    "proxy_realised_bin": realised_bin,
    "model_brier_multiclass": model_brier_multiclass,
    "market_brier_multiclass": market_brier_multiclass,
    "model_log_score": model_log_score,
    "market_log_score": market_log_score,
}])

score_summary

,event_slug,event_date,comparison_time_utc,selected_forecast_lead_time_days,forecast_mu,assumed_sigma,realised_proxy_temperature,proxy_realised_bin,model_brier_multiclass,market_brier_multiclass,model_log_score,market_log_score
0,highest-temperature-in-hong-kong-on-june-10-2026,2026-06-10,2026-06-09 12:00:00+00:00,1,28.2,1.5,27.4,"[27.0, 28.0)",0.71451,1.036264,1.447704,2.38717


## 8. Sensitivity to assumed forecast uncertainty

This sensitivity check shows how model-implied probabilities change when the assumed forecast uncertainty changes.

The final dissertation should estimate this uncertainty rather than choosing it manually.

In [26]:
sigma_grid = [0.75, 1.0, 1.5, 2.0, 2.5]

sensitivity_rows = []

for sig in sigma_grid:
    probs = []
    for _, row in market_prob_df.iterrows():
        probs.append(bin_probability_normal(row["bin_lower"], row["bin_upper"], mu, sig))
    
    prob_sum = sum([p for p in probs if p is not None])
    probs_norm = [p / prob_sum if prob_sum and prob_sum > 0 else None for p in probs]
    
    for i, row in market_prob_df.reset_index(drop=True).iterrows():
        sensitivity_rows.append({
            "sigma": sig,
            "bin_label": row["bin_label"],
            "model_probability_normalised": probs_norm[i],
        })

sensitivity_df = pd.DataFrame(sensitivity_rows)

sensitivity_pivot = sensitivity_df.pivot(
    index="bin_label",
    columns="sigma",
    values="model_probability_normalised",
)

sensitivity_pivot

sigma,0.75,1.00,1.50,2.00,2.50
bin_label,,,,,
< 25.0,9.920764e-06,6.871379e-04,0.016449,0.054799,0.100273
>= 34.0,5.218048e-15,3.315746e-09,0.000055,0.001866,0.010170
"[25.0, 26.0)",1.666797e-03,1.321631e-02,0.054785,0.080867,0.089157
"[26.0, 27.0)",5.312257e-02,1.011662e-01,0.140622,0.138587,0.126184
"[27.0, 28.0)",3.400636e-01,3.056706e-01,0.235109,0.185919,0.152505
"[28.0, 29.0)",4.620759e-01,3.674043e-01,0.256134,0.195250,0.157397
"[29.0, 30.0)",1.348637e-01,1.759251e-01,0.181832,0.160518,0.138722
"[30.0, 31.0)",8.103055e-03,3.337519e-02,0.084096,0.103303,0.104406
"[31.0, 32.0)",9.427870e-05,2.482782e-03,0.025325,0.052040,0.067101


## 9. Save lightweight local outputs

These files are saved locally for inspection but are ignored by git.

In [28]:
os.makedirs("../data/processed/mini_pipeline", exist_ok=True)

yes_df.to_csv("../data/processed/mini_pipeline/hk_june10_yes_bins.csv", index=False)
market_prob_df.to_csv("../data/processed/mini_pipeline/hk_june10_market_probs_at_time.csv", index=False)
weather_forecast_df.to_csv("../data/processed/mini_pipeline/hk_june10_weather_forecasts.csv", index=False)
comparison_pipeline_df.to_csv("../data/processed/mini_pipeline/hk_june10_model_vs_market_comparison.csv", index=False)
score_summary.to_csv("../data/processed/mini_pipeline/hk_june10_score_summary.csv", index=False)
sensitivity_df.to_csv("../data/processed/mini_pipeline/hk_june10_sigma_sensitivity.csv", index=False)

print("Saved local mini-pipeline outputs. These files are ignored by git.")

Saved local mini-pipeline outputs. These files are ignored by git.


## Current findings

This notebook creates an initial end-to-end mini-pipeline for the dissertation.

Current result:

- Polymarket temperature-bin markets are retrieved from the Gamma API.
- YES prices and CLOB token IDs are extracted for each temperature bin.
- CLOB historical prices are retrieved using `interval="max"`.
- A comparison time is chosen approximately one day before settlement.
- Market-implied probabilities are extracted at that comparison time and normalised across bins.
- Open-Meteo lead-time forecasts are retrieved for the same event date.
- A simple normal predictive distribution is built using one lead-time forecast as the mean.
- The predictive distribution is mapped into Polymarket temperature-bin probabilities.
- Model-implied and market-implied probabilities are compared in a discrepancy table.
- A proxy realised-bin check and simple Brier/log-score comparison are included.

Main caveats:

- Open-Meteo is currently a proxy weather source, not the exact Hong Kong Observatory settlement source.
- The assumed normal uncertainty `sigma` is arbitrary and should later be estimated from historical forecast errors.
- The selected market is already resolved, so this notebook is an initial historical prototype rather than a live trading test.
- The full analysis should repeat this process across more markets, cities, lead times and forecast sources.
- The integer-bin convention used here treats labels such as `28°C` as `[28.0, 29.0)`. This is a practical modelling convention for comparing a continuous forecast distribution with discrete market bins, but the exact contract wording and settlement-source convention should be verified before the full empirical analysis.

The core structure is now visible:

`forecast distribution -> model-implied event probabilities -> market-implied probabilities -> discrepancy -> scoring/backtest`